# Stable Audio Open 1.0

https://huggingface.co/stabilityai/stable-audio-open-1.0




In [3]:
!pip uninstall -y torchao
!pip install -q torchsde -U diffusers

import torch
import soundfile as sf
from diffusers import StableAudioPipeline
from huggingface_hub import login
from IPython.display import Audio, display

login()  # paste your HF token (skip if HF_TOKEN is set in Colab secrets)

pipe = StableAudioPipeline.from_pretrained(
    "stabilityai/stable-audio-open-1.0", torch_dtype=torch.float16
).to("cuda")

prompt = "EDM ighasfblvhjadsfgliuasfgiu;asfgiu"
negative_prompt = "Low quality."
generator = torch.Generator("cuda").manual_seed(0)

audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_end_in_s=21.0,
    num_waveforms_per_prompt=3,
    generator=generator,
).audios

sr = pipe.vae.sampling_rate
for i, a in enumerate(audio):
    sf.write(f"output_{i}.wav", a.T.float().cpu().numpy(), sr)
    display(Audio(f"output_{i}.wav"))

Output hidden; open in https://colab.research.google.com to view.

# Stable Audio 3 — Small-Music

Text-to-music with Stability AI’s `stable-audio-3-small-music` (433M parameters, up to 120 s, stereo 44.1 kHz). It runs on CPU, so no GPU runtime is needed (a GPU makes it faster).

**Before running:**
1. Log in to Hugging Face and accept the terms on the [model page](https://huggingface.co/stabilityai/stable-audio-3-small-music). It’s gated, and the terms include Gemma’s (the text encoder is T5Gemma).
2. Create a read token (huggingface.co → Settings → Access Tokens) and add it as a Colab secret named `HF_TOKEN`, or paste it when `login()` asks.
3. Run the install cell. If Colab asks you to **restart the session**, do it, then continue from the next cell.

In [ ]:
# Install the Stable Audio 3 library (may replace Colab's torch with the version it pins)
!pip install -q "git+https://github.com/Stability-AI/stable-audio-3.git" soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 3.7 MB/s eta 0:00:00
 

In [ ]:
import torch
import soundfile as sf
from huggingface_hub import login
from IPython.display import Audio, display
from stable_audio_3 import StableAudioModel

login()  # skip if HF_TOKEN is set in Colab secrets

device = "cuda" if torch.cuda.is_available() else "cpu"
model = StableAudioModel.from_pretrained("small-music", device=device)

No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention


model_config.json:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

t5gemma-b-b-ul2/tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

t5gemma-b-b-ul2/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.18GB            

t5gemma-b-b-ul2/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

In [ ]:
prompt = "weird lions roaring"

out = model.generate(
    prompt=prompt,
    duration=21,    # seconds (max 120 for small-music)
    steps=8,        # default; more steps don't help the post-trained model
    seed=0,         # -1 = random
    batch_size=3,   # three variations
)
# Note: cfg_scale and negative_prompt only affect the “-base” checkpoints.

100%|██████████| 8/8 [00:00<00:00, 17.53it/s]


In [ ]:
# Normalize the output to a (batch, channels, samples) tensor plus sample rate
def unpack(out, default_sr=44100):
    if isinstance(out, (tuple, list)) and len(out) == 2:
        a, b = out
        if isinstance(a, int):
            return b, a
        if isinstance(b, int):
            return a, b
    return out, default_sr

audio, sr = unpack(out)
audio = torch.as_tensor(audio).float().cpu()
if audio.dim() == 2:          # single clip: (channels, samples)
    audio = audio.unsqueeze(0)

for i, clip in enumerate(audio):
    clip = clip / clip.abs().max().clamp(min=1e-8)   # peak normalize
    path = f"sa3_small_music_{i}.wav"
    sf.write(path, clip.T.numpy(), sr)
    print(path, f"{clip.shape[-1] / sr:.1f} s @ {sr} Hz")
    display(Audio(path))